# acai.vector_store — Test & Examples Notebook

Interactive test suite for the hexagonal vector-store module.
Run each cell top-to-bottom. **No real PostgreSQL database is needed** — all tests use a mocked `psycopg2` connection.

**Architecture under test:**

| Layer | Component | Description |
|-------|-----------|-------------|
| Port | `VectorStorePort` | Abstract `upsert` / `search` / `delete` / `close` interface |
| Domain | `VectorStoreConfig` | Configuration value object with validation |
| Domain | `VectorRecord` | Immutable record: id, text, vector, metadata |
| Domain | `SearchResult` | Immutable search result: record + similarity score |
| Domain | Exceptions | `VectorStoreError` hierarchy (Connection, Query, Configuration) |
| Adapter | `PgvectorStore` | PostgreSQL + pgvector (psycopg2) |
| Factory | `create_vector_store()` | Composition root |

## 0 — Setup & Imports

In [ ]:
import sys, json
from pathlib import Path
from unittest.mock import MagicMock, patch, PropertyMock

# Ensure acai is importable
_project_root = Path.cwd()
while _project_root.name != "acai" and _project_root != _project_root.parent:
    _project_root = _project_root.parent
_shared_python = _project_root.parent  # .../shared/python
if str(_shared_python) not in sys.path:
    sys.path.insert(0, str(_shared_python))

from acai.logging import create_logger, LoggerConfig, LogLevel
from acai.ai_vector_store import (
    create_vector_store,
    VectorStorePort,
    VectorStoreConfig,
    VectorRecord,
    SearchResult,
    VectorStoreError,
    ConnectionError,
    QueryError,
    ConfigurationError,
)
from acai.ai_vector_store.adapters.outbound.pgvector_store import (
    PgvectorStore,
    PgvectorConfig,
)

# Shared logger
_logger = create_logger(
    LoggerConfig(service_name="vector-store-test", log_level=LogLevel.DEBUG)
)

# Test result tracker
_results: list[tuple[str, bool, str]] = []


def _record(name: str, passed: bool, detail: str = "") -> None:
    _results.append((name, passed, detail))
    status = "\u2705 PASS" if passed else "\u274c FAIL"
    print(f"{status}  {name}" + (f"  \u2014 {detail}" if detail else ""))


# \u2500\u2500 Mock builder \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n

def _make_store(**overrides) -> PgvectorStore:
    """Build a PgvectorStore with a fully mocked psycopg2 connection.

    No real PostgreSQL instance is needed.
    """
    defaults = {
        "dsn": "host=localhost dbname=testdb",
        "table": "law_embeddings",
        "schema": "app",
        "dimension": 3,
        "distance_metric": "cosine",
    }
    defaults.update(overrides)
    cfg = PgvectorConfig(**defaults)
    with (
        patch("acai.ai_vector_store.adapters.outbound.pgvector_store.psycopg2") as mock_pg,
        patch("acai.ai_vector_store.adapters.outbound.pgvector_store.register_vector"),
    ):
        mock_pg.Error = type("Error", (Exception,), {})
        mock_conn = MagicMock()
        mock_conn.closed = False
        mock_pg.connect.return_value = mock_conn
        store = PgvectorStore(logger=_logger, config=cfg)
    store._conn = mock_conn
    return store


print("Imports OK \u2714")

Imports OK ✔


## 1 — Domain: `VectorStoreConfig` Validation

The config dataclass validates `dimension`, `distance_metric`, and `table` in `__post_init__`.

In [4]:
# Default values
cfg = VectorStoreConfig()
_record("Config — defaults", cfg.dimension == 1024 and cfg.distance_metric == "cosine")

# Custom values
cfg = VectorStoreConfig(table="my_vectors", schema="app", dimension=768, distance_metric="l2")
_record("Config — custom", cfg.dimension == 768 and cfg.distance_metric == "l2")

# Invalid dimension
try:
    VectorStoreConfig(dimension=0)
    _record("Config — zero dimension rejected", False)
except ConfigurationError as e:
    _record("Config — zero dimension rejected", True, str(e))

# Invalid distance metric
try:
    VectorStoreConfig(distance_metric="manhattan")
    _record("Config — invalid metric rejected", False)
except ConfigurationError as e:
    _record("Config — invalid metric rejected", True, str(e))

# Empty table
try:
    VectorStoreConfig(table="")
    _record("Config — empty table rejected", False)
except ConfigurationError as e:
    _record("Config — empty table rejected", True, str(e))

✅ PASS  Config — defaults
✅ PASS  Config — custom
✅ PASS  Config — zero dimension rejected  — dimension must be positive
✅ PASS  Config — invalid metric rejected  — distance_metric must be one of {'inner_product', 'cosine', 'l2'}, got 'manhattan'
✅ PASS  Config — empty table rejected  — table must be a non-empty string


## 2 — Domain: `VectorRecord` & `SearchResult`

Both are frozen (immutable) dataclasses.

In [5]:
# VectorRecord
rec = VectorRecord(
    id="sr210::art1",
    text="Das Zivilgesetzbuch tritt am 1. Januar 1912 in Kraft.",
    vector=[0.12, 0.45, 0.78],
    metadata={"law_sr_number": "210", "article_id": "art1", "sachgebiet": "Privatrecht"},
)
_record("VectorRecord — created", rec.id == "sr210::art1")
_record("VectorRecord — metadata", rec.metadata["sachgebiet"] == "Privatrecht")

# Frozen: mutation must fail
try:
    rec.id = "changed"  # type: ignore
    _record("VectorRecord — frozen", False)
except AttributeError:
    _record("VectorRecord — frozen", True)

# Default empty metadata
bare = VectorRecord(id="x", text="t", vector=[1.0])
_record("VectorRecord — default metadata", bare.metadata == {})

# SearchResult
sr = SearchResult(record=rec, score=0.95)
_record("SearchResult — score", sr.score == 0.95)
_record("SearchResult — record preserved", sr.record.id == "sr210::art1")

try:
    sr.score = 0.5  # type: ignore
    _record("SearchResult — frozen", False)
except AttributeError:
    _record("SearchResult — frozen", True)

✅ PASS  VectorRecord — created
✅ PASS  VectorRecord — metadata
✅ PASS  VectorRecord — frozen
✅ PASS  VectorRecord — default metadata
✅ PASS  SearchResult — score
✅ PASS  SearchResult — record preserved
✅ PASS  SearchResult — frozen


## 3 — Exception Hierarchy

All exceptions derive from `VectorStoreError`.

In [6]:
_record("Exceptions — ConnectionError", issubclass(ConnectionError, VectorStoreError))
_record("Exceptions — QueryError", issubclass(QueryError, VectorStoreError))
_record("Exceptions — ConfigurationError", issubclass(ConfigurationError, VectorStoreError))

# Can catch all with base class
try:
    raise QueryError("test")
except VectorStoreError:
    _record("Exceptions — base catch works", True)

✅ PASS  Exceptions — ConnectionError
✅ PASS  Exceptions — QueryError
✅ PASS  Exceptions — ConfigurationError
✅ PASS  Exceptions — base catch works


## 4 — `PgvectorConfig`: DSN Building

The adapter config supports both an explicit DSN string and individual connection fields.

In [7]:
# Explicit DSN wins
cfg = PgvectorConfig(dsn="host=prod-db port=5433 dbname=lawbot")
_record("PgvectorConfig — DSN wins", "prod-db" in cfg.build_dsn())

# Build from fields
cfg = PgvectorConfig(host="aurora.internal", port=5432, dbname="embeddings", user="app", password="s3cret")
dsn = cfg.build_dsn()
_record("PgvectorConfig — build from fields", "aurora.internal" in dsn and "embeddings" in dsn)
print(f"  Built DSN: {dsn}")

# Inherits base validation
try:
    PgvectorConfig(dimension=-1)
    _record("PgvectorConfig — inherits validation", False)
except ConfigurationError:
    _record("PgvectorConfig — inherits validation", True)

✅ PASS  PgvectorConfig — DSN wins
✅ PASS  PgvectorConfig — build from fields
  Built DSN: host=aurora.internal port=5432 dbname=embeddings user=app password=s3cret
✅ PASS  PgvectorConfig — inherits validation


## 5 — Mocked `PgvectorStore`: Upsert Embeddings

Demonstrates the full upsert flow against a **mocked PostgreSQL connection**.
No real database needed — psycopg2 is entirely patched.

In [ ]:
store = _make_store()

# Prepare sample law-article embedding records
records = [
    VectorRecord(
        id="sr210::art1",
        text="Das Recht wird angewandt auf alle Rechtsfragen.",
        vector=[0.12, 0.45, 0.78],
        metadata={"law_sr_number": "210", "law_name": "ZGB", "article_id": "art1"},
    ),
    VectorRecord(
        id="sr220::art1",
        text="Wer einem andern Schaden zufügt, ist zum Ersatze verpflichtet.",
        vector=[0.33, 0.55, 0.11],
        metadata={"law_sr_number": "220", "law_name": "OR", "article_id": "art1"},
    ),
    VectorRecord(
        id="sr311::art1",
        text="Dem Strafgesetze ist unterworfen, wer in der Schweiz ein Verbrechen begeht.",
        vector=[0.90, 0.10, 0.05],
        metadata={"law_sr_number": "311", "law_name": "StGB", "article_id": "art1"},
    ),
]

# Mock execute_batch so we can inspect the call
with patch("acai.ai_vector_store.adapters.outbound.pgvector_store.psycopg2.extras") as mock_extras:
    count = store.upsert(records)

_record("Upsert — returns record count", count == 3, f"count={count}")
_record("Upsert — commit called", store._conn.commit.called)

# Empty list returns 0 without hitting the DB
store._conn.reset_mock()
_record("Upsert — empty list returns 0", store.upsert([]) == 0)
_record("Upsert — no commit for empty", not store._conn.commit.called)

AttributeError: module 'acai' has no attribute 'vector_store'

## 6 — Mocked `PgvectorStore`: Similarity Search

Simulates a cosine-similarity search returning pre-configured rows from the mocked cursor.

In [ ]:
store = _make_store()

# Configure the mock cursor to return two "similar" rows
mock_cursor = MagicMock()
mock_cursor.fetchall.return_value = [
    ("sr210::art1", "Das Recht wird angewandt...", [0.12, 0.45, 0.78], {"law_sr_number": "210"}, 0.05),
    ("sr220::art1", "Wer einem andern Schaden...", [0.33, 0.55, 0.11], {"law_sr_number": "220"}, 0.23),
]
store._conn.cursor.return_value.__enter__ = MagicMock(return_value=mock_cursor)

# Search
query_vector = [0.10, 0.44, 0.77]
results = store.search(query_vector, top_k=2)

_record("Search — returns 2 results", len(results) == 2)
_record("Search — first is closest", results[0].score < results[1].score)
_record("Search — result is SearchResult", isinstance(results[0], SearchResult))
_record("Search — record has id", results[0].record.id == "sr210::art1")
_record("Search — metadata preserved", results[0].record.metadata["law_sr_number"] == "210")

# Inspect the SQL that was executed
sql_executed = mock_cursor.execute.call_args[0][0]
_record("Search — uses cosine operator", "<=>" in sql_executed, "operator: <=>")
_record("Search — targets correct table", "app.law_embeddings" in sql_executed)

print(f"\nResults:")
for r in results:
    print(f"  {r.record.id}  score={r.score:.4f}  text={r.record.text!r}")

2026-03-24 17:21:48,547 - INFO - Connected to PostgreSQL (pgvector)


✅ PASS  Search — returns 2 results
✅ PASS  Search — first is closest
✅ PASS  Search — result is SearchResult
✅ PASS  Search — record has id
✅ PASS  Search — metadata preserved
✅ PASS  Search — uses cosine operator  — operator: <=>
✅ PASS  Search — targets correct table

Results:
  sr210::art1  score=0.0500  text='Das Recht wird angewandt...'
  sr220::art1  score=0.2300  text='Wer einem andern Schaden...'


## 7 — Mocked Search with Metadata Filter

Filter on JSONB metadata columns (e.g. only search within SR 210).

In [ ]:
store = _make_store()

mock_cursor = MagicMock()
mock_cursor.fetchall.return_value = [
    ("sr210::art1", "Das Recht...", [0.12, 0.45, 0.78], {"law_sr_number": "210"}, 0.05),
]
store._conn.cursor.return_value.__enter__ = MagicMock(return_value=mock_cursor)

results = store.search(
    [0.1, 0.4, 0.8],
    top_k=5,
    filter_metadata={"law_sr_number": "210"},
)

sql = mock_cursor.execute.call_args[0][0]
params = mock_cursor.execute.call_args[0][1]

_record("Filtered search — WHERE clause present", "WHERE" in sql)
_record("Filtered search — metadata key in params", "210" in str(params))
_record("Filtered search — 1 result returned", len(results) == 1)

print(f"\nGenerated SQL (excerpt):")
for line in sql.strip().splitlines():
    print(f"  {line.strip()}")

2026-03-24 17:21:53,623 - INFO - Connected to PostgreSQL (pgvector)


✅ PASS  Filtered search — WHERE clause present
✅ PASS  Filtered search — metadata key in params
✅ PASS  Filtered search — 1 result returned

Generated SQL (excerpt):
  SELECT id, text, embedding, metadata,
  embedding <=> %(query_vec)s AS distance
  FROM   app.law_embeddings
  WHERE metadata->>%(fk_0)s = %(fv_0)s
  ORDER BY distance
  LIMIT  %(top_k)s;


## 8 — Mocked Delete

Delete records by their external IDs.

In [ ]:
store = _make_store()

# Empty delete
_record("Delete — empty list returns 0", store.delete([]) == 0)

# Mock cursor.rowcount to simulate 2 deleted rows
mock_cursor = MagicMock()
mock_cursor.rowcount = 2
store._conn.cursor.return_value.__enter__ = MagicMock(return_value=mock_cursor)

deleted = store.delete(["sr210::art1", "sr220::art1"])

sql = mock_cursor.execute.call_args[0][0]
_record("Delete — SQL uses ANY()", "ANY" in sql)
_record("Delete — commit called", store._conn.commit.called)

2026-03-24 17:21:58,358 - INFO - Connected to PostgreSQL (pgvector)
2026-03-24 17:21:58,359 - INFO - Deleted 2 records


✅ PASS  Delete — empty list returns 0
✅ PASS  Delete — SQL uses ANY()
✅ PASS  Delete — commit called


## 9 — Context Manager & Rollback

The adapter supports `with` blocks. On exception, it rolls back; on clean exit, it closes.

In [ ]:
# Clean exit
store = _make_store()
with store:
    pass
_record("Context manager — close on clean exit", store._conn.close.called)

# Exception → rollback
store = _make_store()
try:
    with store:
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass
_record("Context manager — rollback on exception", store._conn.rollback.called)
_record("Context manager — close after rollback", store._conn.close.called)

2026-03-24 17:22:03,245 - INFO - Connected to PostgreSQL (pgvector)
2026-03-24 17:22:03,246 - INFO - PostgreSQL connection closed
2026-03-24 17:22:03,247 - INFO - Connected to PostgreSQL (pgvector)
2026-03-24 17:22:03,248 - INFO - PostgreSQL connection closed


✅ PASS  Context manager — close on clean exit
✅ PASS  Context manager — rollback on exception
✅ PASS  Context manager — close after rollback


## 10 — Factory: `create_vector_store()`

The factory wires the adapter from configuration.

In [ ]:
# pgvector provider (mocked)
with (
    patch("acai.ai_vector_store.adapters.outbound.pgvector_store.psycopg2") as mock_pg,
    patch("acai.ai_vector_store.adapters.outbound.pgvector_store.register_vector"),
):
    mock_pg.Error = type("Error", (Exception,), {})
    mock_pg.connect.return_value = MagicMock()
    store = create_vector_store(
        _logger,
        provider="pgvector",
        dsn="host=localhost dbname=test",
        dimension=3,
    )

_record("Factory — returns VectorStorePort", isinstance(store, VectorStorePort))
_record("Factory — concrete is PgvectorStore", isinstance(store, PgvectorStore))

# Unknown provider
try:
    create_vector_store(_logger, provider="pinecone")
    _record("Factory — unknown provider rejected", False)
except ConfigurationError as e:
    _record("Factory — unknown provider rejected", True, str(e))

2026-03-24 17:22:07,666 - INFO - Connected to PostgreSQL (pgvector)


✅ PASS  Factory — returns VectorStorePort
✅ PASS  Factory — concrete is PgvectorStore
✅ PASS  Factory — unknown provider rejected  — Unknown provider 'pinecone'. Choose from: pgvector


## 11 — End-to-End Mock: Full Pipeline Flow

Simulates the complete law-bot workflow: create records → upsert → search → delete. All against a mocked PostgreSQL.

In [ ]:
store = _make_store()

# ── Step 1: Build records from pipeline output ──
pipeline_output = [
    {
        "embedding_text": "Art. 1 ZGB: Das Recht wird angewandt.",
        "embedding_vector": [0.12, 0.45, 0.78],
        "model": "voyage-3-large",
        "metadata": {"law_sr_number": "210", "law_name": "ZGB", "article_id": "art1"},
    },
    {
        "embedding_text": "Art. 1 OR: Vertrag kommt zustande.",
        "embedding_vector": [0.33, 0.55, 0.11],
        "model": "voyage-3-large",
        "metadata": {"law_sr_number": "220", "law_name": "OR", "article_id": "art1"},
    },
]

# Map pipeline output → VectorRecord (this mapping lives in the application layer)
records = [
    VectorRecord(
        id=f"{item['metadata']['law_sr_number']}::{item['metadata']['article_id']}",
        text=item["embedding_text"],
        vector=item["embedding_vector"],
        metadata=item["metadata"],
    )
    for item in pipeline_output
]

_record("E2E — mapped 2 records", len(records) == 2)

# ── Step 2: Upsert ──
with patch("acai.ai_vector_store.adapters.outbound.pgvector_store.psycopg2.extras") as mock_extras:
    count = store.upsert(records)
_record("E2E — upserted", count == 2)

# ── Step 3: Search (chatbot query) ──
mock_cursor = MagicMock()
mock_cursor.fetchall.return_value = [
    ("210::art1", "Art. 1 ZGB: Das Recht wird angewandt.", [0.12, 0.45, 0.78], {"law_sr_number": "210"}, 0.02),
]
store._conn.cursor.return_value.__enter__ = MagicMock(return_value=mock_cursor)

hits = store.search([0.11, 0.44, 0.77], top_k=1)
_record("E2E — search found closest", hits[0].record.text.startswith("Art. 1 ZGB"))
_record("E2E — score is float", isinstance(hits[0].score, float))

# ── Step 4: Delete obsolete ──
mock_cursor2 = MagicMock()
mock_cursor2.rowcount = 1
store._conn.cursor.return_value.__enter__ = MagicMock(return_value=mock_cursor2)
store.delete(["220::art1"])
_record("E2E — delete called", mock_cursor2.execute.called)

print("\n\u2714 Full pipeline flow completed (mocked)")

2026-03-24 17:22:12,132 - INFO - Connected to PostgreSQL (pgvector)
2026-03-24 17:22:12,134 - INFO - Upserted 2 records
2026-03-24 17:22:12,135 - INFO - Deleted 1 records


✅ PASS  E2E — mapped 2 records
✅ PASS  E2E — upserted
✅ PASS  E2E — search found closest
✅ PASS  E2E — score is float
✅ PASS  E2E — delete called

✔ Full pipeline flow completed (mocked)


## Summary

In [ ]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total  = len(_results)

print(f"\n{'='*60}")
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print(f"{'='*60}\n")

for name, ok, detail in _results:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}" + (f"  \u2014 {detail}" if detail else ""))


  RESULTS: 47/47 passed, 0 failed

  [PASS] Config — defaults
  [PASS] Config — custom
  [PASS] Config — zero dimension rejected  — dimension must be positive
  [PASS] Config — invalid metric rejected  — distance_metric must be one of {'inner_product', 'l2', 'cosine'}, got 'manhattan'
  [PASS] Config — empty table rejected  — table must be a non-empty string
  [PASS] VectorRecord — created
  [PASS] VectorRecord — metadata
  [PASS] VectorRecord — frozen
  [PASS] VectorRecord — default metadata
  [PASS] SearchResult — score
  [PASS] SearchResult — record preserved
  [PASS] SearchResult — frozen
  [PASS] Exceptions — ConnectionError
  [PASS] Exceptions — QueryError
  [PASS] Exceptions — ConfigurationError
  [PASS] Exceptions — base catch works
  [PASS] PgvectorConfig — DSN wins
  [PASS] PgvectorConfig — build from fields
  [PASS] PgvectorConfig — inherits validation
  [PASS] Upsert — returns record count  — count=3
  [PASS] Upsert — commit called
  [PASS] Upsert — empty list returns 0
  